# Predict Parent Motivation — Few-Shot LLM (Batch API)

Use Claude to classify `parent_motivation` (8 nominal classes) from highlight text +
scenario context, with k-shot examples drawn from the training fold.

**Design**:
- Manual 5-fold CV (stratified where possible)
- k-shot examples drawn from *training fold only* (no leakage)
- Stratified sampling: k//8 examples per class (ensures each class represented)
- Shot counts tested: k=8, 16, 24 (scales k//8 examples per class)
- All k values submitted as a **single batch** — ~3× cost saving vs sequential
- Cache keys: `motiv_k{k}_fold{fold}_pos{row_id}` — never overwrite existing keys

**Workflow** (4 phases):
1. **Prepare** — build all batch requests for uncached rows
2. **Submit** — send batch, save batch ID to `motivation_batch_ids.json`
3. **Poll** — wait for completion, write results to shared cache
4. **Evaluate** — read from cache, compute metrics (re-runnable without new API calls)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.metrics import f1_score, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

try:
    import anthropic
    from dotenv import dotenv_values
    HAS_ANTHROPIC = True
except ImportError:
    HAS_ANTHROPIC = False
    print('anthropic / dotenv not installed — Claude cells will be skipped.')

DATA_DIR   = Path('../../data-exports/20260412_183830')
OUTPUT_DIR = DATA_DIR / 'highlight_analysis_output'
OUT_DIR    = OUTPUT_DIR / 'motivation_classifier_output'
OUT_DIR.mkdir(exist_ok=True)

MOD_FILE   = DATA_DIR / 'moderation_sessions_export_20260412_183830.csv'

CACHE_FILE     = OUT_DIR / 'motivation_llm_cache.json'
BATCH_IDS_FILE = OUT_DIR / 'motivation_batch_ids.json'

RANDOM_STATE = 42
OUTER_K      = 5
K_VALUES     = [8, 16, 24]
CLAUDE_MODEL = 'claude-opus-4-7'
POLL_INTERVAL = 30

## Load data

In [ ]:
df_sel = pd.read_csv(OUTPUT_DIR / 'df_sel.csv')

r5 = pd.read_csv(DATA_DIR / 'R5_highlights_coded', sep='\t')
r5.columns = r5.columns.str.strip()
r5 = r5.rename(columns={
    'highlight_id':      'selection_id',
    'Parent Motivation': 'r5_motivation',
    'Model Strategy':    'r5_strategy',
})
r5['r5_motivation'] = r5['r5_motivation'].replace({
    '': None, 'null': None,
    'Response Could Evoke Strong Emotion': 'Response Could Evoke Strong Emotions',
    'Parents Trust of Model Capabilities': None,
})
r5 = r5[r5['r5_motivation'].notna()]

# Join scenario text
mod = pd.read_csv(MOD_FILE)
scenario_text = mod.drop_duplicates('scenario_id')[['scenario_id', 'scenario_prompt', 'original_response']]

df = (
    df_sel
    .merge(r5[['selection_id', 'r5_motivation']], on='selection_id', how='inner')
    .drop(columns=['parent_motivation'], errors='ignore')
    .rename(columns={'r5_motivation': 'parent_motivation'})
    .merge(scenario_text, on='scenario_id', how='left')
    .reset_index(drop=True)
)

print(f'Shape: {df.shape}')
assert df['scenario_prompt'].isna().sum() == 0, 'Missing scenario_prompt!'
assert df['original_response'].isna().sum() == 0, 'Missing original_response!'
print(df['parent_motivation'].value_counts().to_string())

In [ ]:
le = LabelEncoder()
y  = le.fit_transform(df['parent_motivation'])
CLASSES   = le.classes_
N_CLASSES = len(CLASSES)

print(f'{N_CLASSES} classes:')
for i, cls in enumerate(CLASSES):
    print(f'  [{i}] {cls}')

## Claude client and cache helpers

In [ ]:
SKIP_CLAUDE = True

if HAS_ANTHROPIC:
    try:
        env      = dotenv_values(Path('.') / '.env')
        api_key  = env.get('CLAUDE_API_KEY')
        base_url = env.get('CLAUDE_API_BASE_URL')
        if api_key:
            client = anthropic.Anthropic(api_key=api_key, base_url=base_url)
            SKIP_CLAUDE = False
            print(f'Client ready. Model: {CLAUDE_MODEL}')
        else:
            print('WARNING: CLAUDE_API_KEY not found in .env — skipping Claude API cells.')
    except Exception as e:
        print(f'Client init error: {e}')


def load_cache() -> dict:
    return json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}

def save_cache(c: dict) -> None:
    CACHE_FILE.write_text(json.dumps(c, indent=2))

def load_batch_ids() -> dict:
    return json.loads(BATCH_IDS_FILE.read_text()) if BATCH_IDS_FILE.exists() else {}

def save_batch_ids(b: dict) -> None:
    BATCH_IDS_FILE.write_text(json.dumps(b, indent=2))


cache = load_cache()
print(f'Existing cache entries: {len(cache)}')

## System prompt and message builders

In [ ]:
SYSTEM_MOTIVATION = """You are an expert researcher studying parental oversight of AI chatbots used by children.
A parent reviewed an AI chatbot interaction with their child and highlighted a specific passage.
Your task is to classify WHY the parent flagged that passage.

Choose EXACTLY ONE of these 8 motivation classes:
  Response Usefulness
  Response Risk Awareness
  Response Could Evoke Strong Emotions
  Response Identification of the Root Cause
  Child Intentions
  Response Complexity
  Response Organization
  Response Confirmation / Contradiction

Class meanings:
- Response Usefulness: parent cares whether the AI response was helpful or appropriate
- Response Risk Awareness: parent notices how the AI handles safety, danger, or harm
- Response Could Evoke Strong Emotions: parent flags emotionally charged or provocative content
- Response Identification of the Root Cause: AI addresses the underlying issue behind the child's question
- Child Intentions: parent focuses on what the child was really asking or trying to do
- Response Complexity: parent notices whether the response difficulty suits the child
- Response Organization: parent notices response structure, clarity, or flow
- Response Confirmation / Contradiction: response confirms or contradicts something the parent knows

You will be given labeled examples, followed by the case to classify.

Respond with ONLY the exact class label string, nothing else.
Example valid response: Response Usefulness"""


def make_case_block(row: pd.Series) -> str:
    return (
        f"AGE BAND: {row.get('age_band', 'unknown')}\n"
        f"DOMAIN: {row.get('domain', 'unknown')} / {row.get('subdomain', 'unknown')}\n\n"
        f"CHILD'S MESSAGE:\n{row['scenario_prompt']}\n\n"
        f"AI RESPONSE:\n{row['original_response']}\n\n"
        f"HIGHLIGHTED TEXT:\n{row['highlight_text']}"
    )


def make_fewshot_prefix(df_train: pd.DataFrame, k: int, random_state: int = 42) -> str:
    """k//8 examples per class (stratified); remaining drawn randomly from majority."""
    per_class = max(1, k // N_CLASSES)
    examples  = []
    for cls_label in CLASSES:
        pool = df_train[df_train['parent_motivation'] == cls_label]
        n    = min(per_class, len(pool))
        if n > 0:
            examples.append(pool.sample(n=n, replace=False, random_state=random_state))
    examples_df = pd.concat(examples).sample(frac=1, random_state=random_state)

    parts = ["EXAMPLES (use these to calibrate your classification):\n"]
    for i, (_, row) in enumerate(examples_df.iterrows(), 1):
        parts.append(f"[EXAMPLE {i} — TRUE MOTIVATION: {row['parent_motivation']}]\n"
                     + make_case_block(row))
    parts.append("\nNow classify the following:")
    return "\n\n".join(parts)

## Phase 1 — Prepare batch requests

Builds batch requests for all uncached (k, fold, row) combinations.

In [ ]:
def get_cv_splits(y, k=OUTER_K):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
    try:
        splits = list(skf.split(np.zeros(len(y)), y))
        return splits
    except ValueError:
        kf = KFold(n_splits=k, shuffle=True, random_state=RANDOM_STATE)
        return list(kf.split(np.zeros(len(y))))


def prepare_motivation_requests(
    df: pd.DataFrame, y: np.ndarray,
    k_values: list[int], cache: dict,
) -> list[dict]:
    splits   = get_cv_splits(y)
    requests = []

    for k in k_values:
        for fold_idx, (train_idx, test_idx) in enumerate(splits):
            prefix  = make_fewshot_prefix(df.iloc[train_idx], k,
                                           random_state=fold_idx * 100 + k)
            df_test = df.iloc[test_idx]

            for pos, (_, row) in zip(test_idx, df_test.iterrows()):
                key      = f'motiv_k{k}_fold{fold_idx}_pos{pos}'
                user_msg = prefix + '\n\n' + make_case_block(row)

                if key not in cache:
                    requests.append({
                        'custom_id': key,
                        'params': {
                            'model':      CLAUDE_MODEL,
                            'max_tokens': 32,
                            'system': [{
                                'type':          'text',
                                'text':          SYSTEM_MOTIVATION,
                                'cache_control': {'type': 'ephemeral'},
                            }],
                            'messages': [{'role': 'user', 'content': user_msg}],
                        },
                    })
    return requests


if not SKIP_CLAUDE:
    cache    = load_cache()
    requests = prepare_motivation_requests(df, y, K_VALUES, cache)
    total    = len(K_VALUES) * OUTER_K * len(df)
    print(f'Requests to submit: {len(requests)}  (already cached: {total - len(requests)})')
else:
    print('SKIP_CLAUDE=True — load cache and evaluate in Phase 4.')

## Phase 2 — Submit batch

In [ ]:
if not SKIP_CLAUDE and requests:
    batch     = client.messages.batches.create(requests=requests)
    batch_id  = batch.id
    batch_ids = load_batch_ids()
    batch_ids['motivation'] = batch_id
    save_batch_ids(batch_ids)
    print(f'Batch submitted: {batch_id}  ({len(requests)} requests)')
    print(f'Status: {batch.processing_status}')
elif not SKIP_CLAUDE:
    print('All requests already cached — skipping submission.')
    batch_id = load_batch_ids().get('motivation')

## Phase 3 — Poll until complete, write results to cache

Re-runnable: recovers batch_id from `motivation_batch_ids.json` if not in scope.

In [ ]:
if not SKIP_CLAUDE:
    if 'batch_id' not in dir() or batch_id is None:
        batch_id = load_batch_ids().get('motivation')
        if not batch_id:
            raise RuntimeError('No batch_id found — run the submit cell first.')
        print(f'Recovered batch_id from file: {batch_id}')

    print(f'Polling {batch_id} every {POLL_INTERVAL}s…')
    while True:
        batch  = client.messages.batches.retrieve(batch_id)
        counts = batch.request_counts
        print(f'  {batch.processing_status} — '
              f'processing={counts.processing}  '
              f'succeeded={counts.succeeded}  '
              f'errored={counts.errored}')
        if batch.processing_status == 'ended':
            break
        time.sleep(POLL_INTERVAL)

    cache    = load_cache()
    n_ok, n_err = 0, 0
    for result in client.messages.batches.results(batch_id):
        if result.result.type == 'succeeded':
            raw = result.result.message.content[0].text.strip()
            cache[result.custom_id] = raw
            n_ok += 1
        else:
            cache[result.custom_id] = None
            n_err += 1

    save_cache(cache)
    print(f'\nCache updated: {n_ok} succeeded, {n_err} failed.  Total: {len(cache)}')

## Phase 4 — Extract predictions and evaluate

In [ ]:
VALID_CLASSES = set(CLASSES)


def extract_predictions(
    df: pd.DataFrame, y: np.ndarray,
    k: int, cache: dict,
) -> np.ndarray:
    """Read class-label predictions from cache. Falls back to majority class on missing."""
    majority = CLASSES[np.bincount(y).argmax()]
    splits   = get_cv_splits(y)
    preds    = np.array([majority] * len(df), dtype=object)

    for fold_idx, (_, test_idx) in enumerate(splits):
        for pos in test_idx:
            raw = cache.get(f'motiv_k{k}_fold{fold_idx}_pos{pos}')
            if raw and raw in VALID_CLASSES:
                preds[pos] = raw

    return le.transform(preds)


def top2_acc(y_true, y_proba) -> float:
    top2 = np.argsort(y_proba, axis=1)[:, -2:]
    return float(np.mean([y_true[i] in top2[i] for i in range(len(y_true))]))


cache  = load_cache()
rows   = []
for k in K_VALUES:
    n_cached = sum(1 for key in cache if key.startswith(f'motiv_k{k}_'))
    if n_cached == 0:
        print(f'k={k}: no cache entries — skipping.')
        continue
    preds = extract_predictions(df, y, k, cache)
    rows.append({
        'k':           k,
        'n_cached':    n_cached,
        'f1_weighted': round(f1_score(y, preds, average='weighted', zero_division=0), 3),
        'f1_macro':    round(f1_score(y, preds, average='macro',    zero_division=0), 3),
    })

if rows:
    results = pd.DataFrame(rows)
    print('Few-shot motivation prediction results:')
    print(results.to_string(index=False))
    results.to_csv(OUT_DIR / 'motivation_cv_fewshot.csv', index=False)
else:
    print('No results yet.  Run Phase 2–3 to submit and retrieve predictions.')

## Confusion matrix — best k

In [ ]:
if rows:
    best_k = results.loc[results['f1_weighted'].idxmax(), 'k']
    print(f'Best k by weighted F1: {best_k}')

    best_preds = extract_predictions(df, y, best_k, cache)
    short_labels = [c.replace('Response ', 'R.').replace('Child Intentions', 'Child Int.')
                    for c in CLASSES]

    fig, ax = plt.subplots(figsize=(11, 9))
    ConfusionMatrixDisplay.from_predictions(
        y, best_preds, display_labels=short_labels, ax=ax,
        xticks_rotation=45,
    )
    ax.set_title(f'Few-shot Claude motivation classifier (k={best_k})', fontsize=11)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'motivation_fewshot_confusion_matrix.png', dpi=100)
    plt.show()
else:
    print('No predictions yet — run Phase 2–3 first.')

## Per-class F1

In [ ]:
if rows:
    per_class_rows = []
    for k in K_VALUES:
        if not any(r['k'] == k for r in rows):
            continue
        preds = extract_predictions(df, y, k, cache)
        per_f1 = f1_score(y, preds, average=None, zero_division=0)
        for cls, f1_val, n in zip(CLASSES, per_f1, np.bincount(y)):
            per_class_rows.append({'k': k, 'class': cls, 'n': n, 'f1': round(f1_val, 3)})

    per_class_df = pd.DataFrame(per_class_rows)
    per_class_df.to_csv(OUT_DIR / 'motivation_fewshot_per_class_f1.csv', index=False)

    pivot = per_class_df.pivot(index='class', columns='k', values='f1')
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot.plot.bar(ax=ax, width=0.7)
    ax.set_title('Per-class F1 by k (few-shot motivation classifier)')
    ax.set_xlabel('Motivation class')
    ax.set_ylabel('F1')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='k')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'motivation_fewshot_per_class_f1.png', dpi=100)
    plt.show()
else:
    print('No predictions yet — run Phase 2–3 first.')

## Consolidated comparison with structured baselines

In [ ]:
struct_file = OUT_DIR / 'motivation_cv_universal.csv'
consolidated_rows = []

if struct_file.exists():
    struct = pd.read_csv(struct_file)
    for _, r in struct.iterrows():
        if pd.isna(r.get('error', None)):
            consolidated_rows.append({
                'approach': f"Structured: {r['model']}",
                'f1_weighted': round(r.get('f1_weighted_mean', np.nan), 3),
                'f1_macro':    round(r.get('f1_macro_mean', np.nan), 3),
            })

for k in K_VALUES:
    if not any(r['k'] == k for r in rows):
        continue
    preds = extract_predictions(df, y, k, cache)
    consolidated_rows.append({
        'approach':    f'Few-shot Claude (k={k})',
        'f1_weighted': round(f1_score(y, preds, average='weighted', zero_division=0), 3),
        'f1_macro':    round(f1_score(y, preds, average='macro',    zero_division=0), 3),
    })

if consolidated_rows:
    consolidated = pd.DataFrame(consolidated_rows)
    print(consolidated.to_string(index=False))
    consolidated.to_csv(OUT_DIR / 'motivation_consolidated.csv', index=False)
else:
    print('Run 1_motivation_baseline.ipynb first for structured results.')